In [ ]:
# ============================================================
# CELL 0: ENVIRONMENT AND COMMIT PROVENANCE
# ============================================================
import sys, platform, subprocess, pulp, pytest

print("Python :", sys.version.split()[0], "|", platform.platform())
print("PuLP   :", pulp.__version__)
print("pytest :", pytest.__version__)

# CBC version (the solver bundled with PuLP)
cbc = pulp.PULP_CBC_CMD.pulp_cbc_path
out = subprocess.run([cbc, "-quit"], capture_output=True, text=True).stdout
print("CBC    :", next((l.strip() for l in out.splitlines() if "Version" in l), cbc))

# Commit / branch (run from inside the repo)
try:
    rev = lambda *a: subprocess.run(["git", *a], capture_output=True, text=True).stdout.strip()
    print("Commit :", rev("rev-parse", "--short", "HEAD"), "on", rev("rev-parse", "--abbrev-ref", "HEAD"))
except FileNotFoundError:
    print("Commit : git not on PATH")

## Cell 1 — Shared Imports

Loads the core Python utilities and scikit-learn metrics used throughout the notebook.

In [ ]:
# ============================================================
# CELL 1: SHARED IMPORTS
# ============================================================
%matplotlib inline

# Import root = bank_pipeline, data
import os, sys
from pathlib import Path

BANK_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "bank_pipeline"), None
)
if BANK_ROOT is None:
    raise RuntimeError(f"bank_pipeline/ not found above {Path.cwd()}")
os.chdir(BANK_ROOT)
if str(BANK_ROOT) not in sys.path:
    sys.path.insert(0, str(BANK_ROOT))

print(f"📁 import root : {BANK_ROOT}")
print(f"📂 working dir : {Path.cwd()}")
for pkg in ("data", "shared", "glass_pipeline"):
    print(f"   {'✅' if (BANK_ROOT / pkg).is_dir() else '❌ MISSING'} {pkg}/")

# Core
import pandas as pd

## Cell 2 — Build Database and Preprocess

Builds the local database, then applies the leakage-aware bank preprocessing pipeline with leaky features removed.

In [ ]:
# ============================================================
# CELL 2: BUILD DATABASE & PREPROCESS
# ============================================================
from data.preprocessing import BankPreprocessor, build_database

build_database()

preprocessor = BankPreprocessor(drop_leaky=True)
df_processed = preprocessor.fit_transform()

## Cell 2B — Validate Preprocessing

Runs the same preprocessing directly from the raw CSV and verifies that it produces the same dataset as the database-backed pipeline.

In [ ]:
# ============================================================
# CELL 2B: VALIDATING DATA
# ============================================================
df = pd.read_csv("./data/raw/bank-additional-full.csv", sep=";")
df_csv = BankPreprocessor(drop_leaky=True, verbose=False).fit_transform(df)

print(f"Raw CSV       : {df.shape}")
print(f"From database : {df_processed.shape}")
print(f"From CSV      : {df_csv.shape}")
print(f"Identical     : {df_csv.equals(df_processed)}")

## Cell 3 — Inspect Categorical Columns

Identifies categorical features in the raw dataset and prints their observed value counts for data auditing.

In [ ]:
# ============================================================
# CELL 3: EXAMINE CATAGORICAL COLUMNS
# ============================================================

categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

print("📌 CATEGORICAL COLUMNS\n" + "-"*60)
categorical_cols

for col in categorical_cols:
    print(f"\n🔎 Column: {col}")
    print("-" * 40)

    values = (
        df[col]
        .astype(str)
        .value_counts(dropna=False)
        .sort_index()
    )

    print(values)


## Cell 4 — Global Train/Test Split

Creates the shared 80/20 train/test split used across all cascade stages with a fixed random seed for reproducibility.

In [ ]:
# ============================================================
# CELL 4: GLOBAL TRAIN / TEST SPLIT
# ============================================================
from data.global_splits.split_manager import GlobalSplitManager

manager = GlobalSplitManager(test_size=0.20, random_state=42)
GLOBAL_SPLIT = manager.create_split(df_processed, target_col='y')

X_train = GLOBAL_SPLIT['X_train']
X_test = GLOBAL_SPLIT['X_test']
y_train = GLOBAL_SPLIT['y_train']
y_test = GLOBAL_SPLIT['y_test']

## Cell 5 — Feature Engineering (LR Stage 1)

Builds the three Stage 1 features shared by both arms (`LR_FEATURES` = `STAGE1_FEATURES`), from `shared/stage1/feature_engineering.py`.

| Feature | What it is | Uses labels? |
|---|---|---|
| `cellular_downturn` | 1 if contacted by cellular **and** `euribor3m < 1.5` (or both macro conditions hold) | No |
| `subscribe_rate_by_euribor_bin` | Train subscription rate within the row's `euribor3m` quantile bin (20 bins, unsmoothed) | Yes — fitted on train |
| `subscribe_rate_by_dow_month` | Smoothed train subscription rate for the row's day-of-week × month cell (λ = 100) | Yes — fitted on train |

**Names.** PR 39 renamed these features (Stage 1 feature contract 2.0.0); their values are unchanged. Outputs saved before the rename show the earlier names `cellular_crisis`, `euribor3m_local_rate` and `dow_month_encoded`.

**Leakage status.** Both encoders are fitted on `X_train` / `y_train` only; test rows are mapped through the stored bins and cells (no fallbacks on this split). The frames built here are fitted on **all** training rows and serve the final refit and the test transform only. Every cross-validated step of Stage 1 (tuning, out-of-fold predictions, calibration, threshold) refits both encoders inside each fold from the raw rows (`X_train_raw` in Cell 8; PR 36 F3), so no training row's own label reaches its out-of-fold prediction.

**Scope.** `euribor3m` falls almost monotonically over the campaign window, so the two encoded features are effectively period-level subscription rates. Valid under the random split used here; not a forward-looking (deployment) claim.

In [ ]:

# ============================================================
# CELL 5: FEATURE ENGINEERING — LR STAGE 1
# ============================================================
# Source: shared/stage1/feature_engineering.py
#
# Produces (3 columns):
#   LIVE  cellular_downturn             — no target used; safe on any split
#   LIVE  subscribe_rate_by_euribor_bin — fitted on X_train + y_train
#   LIVE  subscribe_rate_by_dow_month   — fitted on X_train + y_train 
# ============================================================
from shared.stage1.feature_engineering import engineer_features

X_train_eng, X_test_eng, engineer = engineer_features(
    X_train, X_test, y_train
)

## Cell 6 — Baseline Logistic Regression

Trains a baseline logistic regression on the 17 pre-engineering features to establish a simple reference point before the constrained Stage 1 pipeline.

- Uses the default `0.50` decision threshold with no probability calibration.
- Reports train/test classification, ranking, and calibration metrics.
- Saves the fitted baseline artifact for later comparison.

In [ ]:
# ============================================================
# CELL 6: BASELINE LR STAGE
# ============================================================
from glass_pipeline.lr.lr_stage import train_baseline_lr_stage

baseline_stage, baseline_metrics = train_baseline_lr_stage(
    X_train, X_test, y_train, y_test,
    save=True, output_dir="./models/lr"
)

## Cell 7 — Correlation Analysis and Pruning

Checks the three engineered Stage 1 features for pairwise multicollinearity before LR tuning.

- Ranks features by absolute correlation with the target.
- Flags feature pairs above the configured `|r| > 0.70` threshold.
- Drops redundant features only when that threshold is exceeded.
- Visualizes the resulting correlation matrix.

In [ ]:
# ============================================================
# CELL 7: CORRELATION ANALYSIS & PRUNING
# ============================================================
from glass_pipeline.lr.correlation_analysis import analyze_and_prune_features

X_train_pruned, X_test_pruned, analyzer = analyze_and_prune_features(
    X_train_eng, X_test_eng, y_train,
    threshold=0.7,
    plot=True
)

    

## Cell 8 — Tuned and Calibrated Stage 1 LR

Trains the production Stage 1 logistic regression on the three engineered features using the full training-only tuning and calibration procedure.

- Tunes LR hyperparameters by cross-validation.
- Fits sigmoid (Platt) calibration, the shared Stage 1 family (PR 39, Step 2A).
- Generates out-of-fold training probabilities.
- Selects the operating threshold from OOF predictions using F2.
- Evaluates the finalized model once on the held-out test set.
- Saves the Stage 1 artifact for downstream cascade use.

The selected threshold is `0.110`, with test ROC-AUC `0.7893`, F2 `0.5597`, and ECE `0.0078`.

In [ ]:
# ============================================================
# CELL 8: LR TUNING + CALIBRATION
# ============================================================
from glass_pipeline.lr.lr_stage_calibrated import train_calibrated_lr_stage

print("=" * 80)
print("📈 STAGE 1: CALIBRATED LOGISTIC REGRESSION")
print("=" * 80)

lr_calibrated_artifact, lr_calibrated_path = train_calibrated_lr_stage(
    X_train_eng, X_test_eng,
    y_train, y_test,
    save=True,
    output_dir="./models/lr",
    X_train_raw=X_train,  # raw rows: target encodings are refit inside each CV fold (PR 36, F3)
    calibration_method="sigmoid",  # the shared Stage 1 family (PR 39, Step 2A)
)

print(f"\n✅ LR Stage 1 complete")
print(f"📦 Saved artifact → {lr_calibrated_path}")
print(f"📊 Test ROC-AUC: {lr_calibrated_artifact.metrics['roc_auc']:.4f}")
print(f"📊 Test F1:      {lr_calibrated_artifact.metrics['f1']:.4f}")
print(f"📊 Test F2:      {lr_calibrated_artifact.metrics['f2']:.4f}")
print(f"📊 ECE:          {lr_calibrated_artifact.metrics['ece']:.4f}")

## Cell 8B — Stage 2 Source

Loads the Stage 2 routers this notebook consumes; nothing here fits Stage 2 (PR 39, execution contract §5.3).

- The accepted PR 38 primary run `c3_20261112_153333`: its GLASS Router bundle (Pass 2 on the OOF Pass 1 remainder in both fits) and RF Router, by pinned path and SHA-256.
- Re-verifies the run's structural acceptance and checks both routers against the Stage 4 router contract, train and test side.
- The legacy notebook Stage 2 (Cells 9–16: the `legacy` profile, Pass 2 on the full population) stays unchanged behind `RUN_LEGACY_STAGE2`, off by default. Set it `True` only to reproduce that historical pathway: it refits Stage 2 and writes the legacy folders.

**Output:** `STAGE2_PINNED`, `glass_path` (read by Cell 19).

In [ ]:
# ============================================================
# CELL 8B: STAGE 2 SOURCE — the accepted PR 38 primary, by pinned path and SHA-256
# ============================================================
# PR 39 (execution contract §5.3): Stage 2 is not fitted in this notebook. Cells 9-16
# (the legacy notebook Stage 2) run only when RUN_LEGACY_STAGE2 is True.
from shared.notebook_switches import register_switch
from shared.pinned_artifacts import load_pinned_stage2

RUN_LEGACY_STAGE2 = False
register_switch("legacy_stage2", "RUN_LEGACY_STAGE2")

if not RUN_LEGACY_STAGE2:
    STAGE2_PINNED = load_pinned_stage2(GLOBAL_SPLIT)
    # F17 primary (G10): Pass 2 fitted on the OOF Pass 1 remainder in both fits
    assert STAGE2_PINNED.stage4_contract["glass"]["pass2_population"] == {
        "cross_fit": "oof_remainder", "full_train": "oof_remainder"}
    glass_path = str(STAGE2_PINNED.glass_path)       # the GLASS Router Cell 19 reads
    print(STAGE2_PINNED.describe())
else:
    print("RUN_LEGACY_STAGE2 = True: Cells 9-16 refit the legacy notebook Stage 2")

## Cell 9 — RF Lift Analysis

Runs an exploratory Random Forest analysis to inspect predictive importance and conversion-rate lift.

- Reports Gini and permutation importance.
- Measures continuous and categorical feature lift relative to the training base rate.
- Visualizes conversion-rate patterns across feature values.

In [ ]:
%%legacy_stage2
# ============================================================
# CELL 9: RF LIFT ANALYSIS 
# ============================================================
# Note: permutation importance inside analyze_rf_lift is scored on X_test.
# Do not use its rankings to change bins or features.
from glass_pipeline.glass_router.rf import analyze_rf_lift

rf_analyzer, rf_analysis = analyze_rf_lift(
    X_train, y_train,
    X_test,  y_test,
    plot=True,
)

## Cell 10 — GLASS Router Feature Engineering

Builds the binary feature representation used by the Stage 2 GLASS Router.

- Constructs 8 continuous source features.
- Encodes them into 29 lift-driven binary bins.
- Applies the same feature transformation to train and test data.
- Validates binary values, mutually exclusive groups, and activation patterns.
- Stores the resulting frames and metadata in `GLASS_ROUTER_DATA`.

In [ ]:
%%legacy_stage2
# ============================================================
# CELL 10: GLASS ROUTER FEATURE ENGINEERING (full train → test)
# ============================================================
from glass_pipeline.glass_router.rf import engineer_features

X_train_eng, X_test_eng = engineer_features(X_train, X_test, y_train)

GLASS_ROUTER_DATA = {
    "X_eng_train":   X_train_eng,
    "y_eng_train":   y_train,
    "X_eng_test":    X_test_eng,
    "y_eng_test":    y_test,
    "feature_names": list(X_train_eng.columns),
}

## Cell 11 — Stage 2 RF Baseline

Trains a fixed-parameter Random Forest on the 29 binary Stage 2 features as a reference model.

- Uses the same engineered feature representation as the GLASS Router.
- Performs no hyperparameter tuning.
- Reports held-out classification, ranking, and calibration metrics.

In [ ]:
%%legacy_stage2
# ============================================================
# CELL 11: RF BASELINE (default params, binary features)
# ============================================================
from glass_pipeline.glass_router.rf import train_rf_baseline

baseline_metrics = train_rf_baseline(
    GLASS_ROUTER_DATA["X_eng_train"], GLASS_ROUTER_DATA["X_eng_test"],
    GLASS_ROUTER_DATA["y_eng_train"], GLASS_ROUTER_DATA["y_eng_test"],
)

## CELL 12 — Stage 2 Optuna Training

Tunes the Random Forest on the 29 engineered binary features with 200 Optuna trials, evaluates it with 10-fold CV, selects an OOF F2 threshold, then refits on the full training set for held-out test evaluation.

`baseline_auc` is used only for comparison reporting.

In [ ]:
%%legacy_stage2
# ============================================================
# CELL 12: RF STAGE 2 — OPTUNA TRAINING
# ============================================================
from glass_pipeline.glass_router.rf import train_rf_stage

rf_result = train_rf_stage(
    GLASS_ROUTER_DATA["X_eng_train"], GLASS_ROUTER_DATA["X_eng_test"],
    GLASS_ROUTER_DATA["y_eng_train"], GLASS_ROUTER_DATA["y_eng_test"],
    n_trials=200,
    baseline_auc=baseline_metrics["roc_auc"],
)

## Cell 12B — RF Diagnostics

Visualizes held-out test performance for the Optuna-tuned Stage 2 RF on the 29 binary features.

- ROC and precision-recall curves.
- Confusion matrix at the OOF-selected F2 operating threshold.

In [ ]:
%%legacy_stage2
# ============================================================
# CELL 12B: RF DIAGNOSTICS (test set)
# ============================================================
from glass_pipeline.glass_router.rf import plot_rf_evaluation

plot_rf_evaluation(
    rf_result,
    GLASS_ROUTER_DATA["X_eng_test"],
    GLASS_ROUTER_DATA["y_eng_test"],
    threshold=rf_result.threshold,
)

## Cell 13 — GLASS Router: Fit and Test Routing

Selects a `CONFIG_PROFILE`, fits GLASS Router on the full training split, and routes the test split.

| Preset | Purpose |
|---|---|
| `legacy` | Reproduces the pre-refactor configuration. |
| `recommended` | Provisional stricter routing configuration. |

Final presets are selected later using **train OOF results only**. The black-box router matches the saved GLASS operating constraints for fair comparison.

**Outputs:** `glass`, `test_out`, `test_proba`.

In [ ]:
%%legacy_stage2
# ============================================================
# CELL 13: GLASS ROUTER — CONFIG + FULL-TRAIN FIT + TEST ROUTING
# ============================================================
import time
from glass_pipeline.glass_router.core.config import GlassRouterConfig
from glass_pipeline.glass_router.pipeline import GlassRouterPipeline
from glass_pipeline.glass_router.rule_generator.rule_logger import RuleLogger
from glass_pipeline.glass_router.core.rule import ABSTAIN
from glass_pipeline.glass_router.model_outputs import build_glass_output

# "legacy"      = previous run's thresholds, now under the union-aware selector
#                 (min_rules lowered; not a parity check against pre-fix runs)
# "recommended" = provisional starting point (see notes below)
CONFIG_PROFILE = "legacy"  # other presets are tried in the comparisons pipeline (PR 38)

# Pass 1 operating target, measured OUT-OF-FOLD in the Stage 2 report.
# The selector's union cap (max_union_leakage_rate_pass1) binds on TRAINING rows,
# and held-out folds run ~1 pt higher, so the cap is set slightly below this.
LEAKAGE_TARGET = 0.15

_SHARED = dict(
    mode="strict",                   # stored only; no code path reads it yet
    max_complexity=3,
    diversity_weight=0.35,           # beam-search reuse penalty + greedy tie-break
    enable_novelty_constraints=True,
    # Depth-2 light pruning: effectively off (0.90) so depth 3 does the filtering
    max_leakage_rate_depth2=0.90,
    max_leakage_fraction_depth2=0.90,
    # Required by GlassRouterConfig but not read by any code path yet
    max_jaccard_overlap=0.5,
    max_high_overlap_rules=20,
    max_feature_reuse_pass1=None,
    max_feature_reuse_pass2=None,
)

_PROFILES = {
    "legacy": dict(
        min_support_pass1=120,   min_support_pass2=25,
        # min_rules lowered from 6 / 5: the set-level novelty floor proved 6 infeasible
        # (100 candidates -> 53 distinct rules). The ILP picks the union-optimal count.
        min_pass1_rules=2,       max_pass1_rules=8,
        min_precision_not_subscribe=0.93, max_precision_not_subscribe=1.00,
        # Per-rule gates: each Pass 1 rule may hold at most 5% / 800 of all subscribers
        max_subscriber_leakage_rate=0.05, max_subscriber_leakage_absolute=800,
        # Pass 1 UNION cap: subscribers routed by the whole Pass 1 rule set
        # (each counted once) <= this share of all subscribers. None = off.
        max_union_leakage_rate_pass1=0.14,
        min_pass2_rules=2,       max_pass2_rules=8,
        min_precision_subscribe=0.20,     max_precision_subscribe=1.00,
        min_recall_subscribe=0.01,        max_recall_subscribe=0.35,
        # Novelty floor (set-level): every selected rule keeps >= this share of its
        # rows that no other selected rule covers
        min_novelty_ratio_pass1=0.20,     min_novelty_ratio_pass2=0.25,
        # lambdas left unset -> ILP defaults 0.15 / 0.08 (tie-break only)
        pass2_population="full",          # original behaviour
    ),
    "recommended": dict(
        # Support: evaluator floor is min(pass1, pass2) // 2 = 25 rows
        min_support_pass1=150,   min_support_pass2=50,

        # Pass 1 — "precision" = share of NON-subscribers in the rule.
        min_pass1_rules=2,       max_pass1_rules=6,
        min_precision_not_subscribe=0.92, max_precision_not_subscribe=1.00,
        # Per-rule gates (each rule) + UNION cap (whole Pass 1 rule set, None = off)
        max_subscriber_leakage_rate=0.10, max_subscriber_leakage_absolute=500,
        max_union_leakage_rate_pass1=0.14,

        # Pass 2 — precision floor 0.30 (~2.7x base rate) blocks the near-universal
        # rules the recall-first score otherwise favours.
        min_pass2_rules=3,       max_pass2_rules=10,
        min_precision_subscribe=0.30,     max_precision_subscribe=1.00,
        min_recall_subscribe=0.01,        max_recall_subscribe=0.25,

        # Novelty floor (set-level): every selected rule keeps >= this share of its
        # rows uniquely (0.01 = nearly off)
        min_novelty_ratio_pass1=0.20,     min_novelty_ratio_pass2=0.20,

        # The evaluator never receives the RF, so every rule gets rf_confidence=0.5
        # and rf_alignment=0.0. With the defaults (0.15 / 0.08) that is a flat
        # -0.155 per selected rule; in the union-aware ILP it only affects the
        # Phase 2 tie-break. 0.0 removes it.
        lambda_rf_uncertainty=0.0,
        lambda_rf_misalignment=0.0,

        # Pass 2 fitted on the rows Pass 1 does not route, using OUT-OF-FOLD Pass 1
        # decisions (5 inner folds) — same construction as the black-box RF router.
        pass2_population="oof_remainder",
        pass1_oof_folds=5,
        random_state=42,
        remainder_min_size=500,
    ),
}

config = GlassRouterConfig(**_SHARED, **_PROFILES[CONFIG_PROFILE])
print(f"Config profile: {CONFIG_PROFILE}\n{config}")
print(f"Pass 1 union leakage cap (train): {config.max_union_leakage_rate_pass1} | "
      f"OOF target (report): {LEAKAGE_TARGET:.0%}")


def make_rule_logger():
    return RuleLogger(
        enabled=False,
        verbose_rejection_logging=False,
        verbose_acceptance_logging=False,
    )


start_time = time.time()

glass = GlassRouterPipeline(
    config=config,
    rf_model=rf_result.model,        # used for feature ordering in rule generation
    rule_logger=make_rule_logger(),
)
# Rules are evaluated on the training rows (X_val = X_train). With
# pass2_population="oof_remainder", Pass 2 is fitted on the out-of-fold Pass 1
# remainder. The cross-fit in Cell 14 repeats exactly this recipe per fold
# (it receives the same `config`, so the union cap applies to every fold).
glass.fit(
    X_train=GLASS_ROUTER_DATA["X_eng_train"],
    y_train=GLASS_ROUTER_DATA["y_eng_train"],
)

test_out = build_glass_output(glass, GLASS_ROUTER_DATA["X_eng_test"], abstain_value=ABSTAIN)
test_proba = glass.predict_proba(GLASS_ROUTER_DATA["X_eng_test"])

elapsed = time.time() - start_time
print(f"\n⏱️  GLASS Router full-train fit + test routing: {elapsed:.1f}s ({elapsed / 60:.1f} min)")

## Cell 14 — Stage 2 Cross-Fit

Generates leakage-safe Stage 2 OOF outputs for Stage 4.

- Repeats the full Stage 2 recipe within each fold.
- Reuses the tuned RF hyperparameters without re-tuning.
- Fits GLASS Router only on each fold's training partition.
- Scores every training row with models that never saw it.
- Preserves fold IDs and OOF RF/GLASS probabilities for downstream stacking.

In [ ]:
%%legacy_stage2
# ============================================================
# CELL 14: STAGE 2 CROSS-FIT — OOF OUTPUTS FOR STAGE 4 
# ============================================================
from glass_pipeline.glass_router.pipeline import crossfit_stage2
 
STAGE_FOLDS = None
 
start_time = time.time()
stage2_oof = crossfit_stage2(
    X_train_raw=X_train,             
    y_train=y_train,
    glass_config=config,
    rf_params=rf_result.params,
    folds=STAGE_FOLDS,
    n_splits=5,
    random_state=42,
    glass_rf_model="fold_rf",        
    rule_logger_factory=make_rule_logger,
)
elapsed = time.time() - start_time
print(f"\n⏱️  Stage 2 cross-fit: {elapsed:.1f}s ({elapsed / 60:.1f} min)")

In [ ]:
# ============================================================
# CELL 14B: GLASS Router selector regression tests
# ============================================================
import subprocess
import sys
from pathlib import Path

BANK_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "glass_pipeline").is_dir())
TEST_FILE = BANK_ROOT / "tests" / "stage2" / "test_ilp_rule_selector_union.py"

result = subprocess.run(
    [sys.executable, "-m", "pytest", str(TEST_FILE), "-q",
     "-p", "no:cacheprovider", "-W", "ignore::DeprecationWarning"],
    cwd=BANK_ROOT,              # makes `import glass_pipeline` resolve to the repo package
    text=True,
    capture_output=True,
)
print(f"python: {sys.executable}\nroot:   {BANK_ROOT}\n")
print(result.stdout[-3000:])    # tail keeps the summary line visible
if result.returncode != 0 and result.stderr:
    print(result.stderr[-3000:])
assert result.returncode == 0, "GLASS Router selector regression tests failed"

## Cell 15 — Stage 2 Report

Compares full-train test behavior with leakage-safe train OOF behavior.

- Reports routing rates, abstention, leakage, recall, and covered metrics.
- Checks the Pass 1 leakage constraint on OOF predictions.
- Measures OOF vs in-sample optimism for GLASS Router and RF.
- Summarizes per-fold stability and selected full-train rules.
- Reports held-out test classification with abstentions mapped to `NOT_SUBSCRIBE`.

The **TRAIN OOF** column is the correct basis for configuration and Stage 4 decisions; test results remain evaluation-only.

In [ ]:
%%legacy_stage2
# ============================================================
# CELL 15: STAGE 2 REPORT — TEST (full-train model) vs OOF (train)
# ============================================================
import pandas as pd
from sklearn.metrics import classification_report
from glass_pipeline.glass_router.core.rule import SUBSCRIBE
from glass_pipeline.glass_router.model_analysis import compute_glass_analysis_metrics
from glass_pipeline.glass_router.pipeline import oof_vs_insample
 
PASS1_LEAKAGE_TARGET = 0.15   # max share of subscribers Pass 1 may route away
 
oof_out = stage2_oof.glass_train_out()
m_test = compute_glass_analysis_metrics(glass, test_out, GLASS_ROUTER_DATA["y_eng_test"], SUBSCRIBE)
m_oof  = compute_glass_analysis_metrics(glass, oof_out,  GLASS_ROUTER_DATA["y_eng_train"], SUBSCRIBE)
 
def _flow(m):
    tp = max(m["total_positives"], 1)
    return {
        "Pass 1 routed":        m["n_pass1"] / m["n_test"],
        "Pass 2 flagged":       m["n_pass2"] / m["n_test"],
        "Abstained":            m["n_uncertain"] / m["n_test"],
        "Pass 1 leakage":       m["blocked_positives"] / tp,
        "Pass 2 recall (all)":  m["overall_recall"],
        "Pass 2 recall (elig)": m["eligible_recall"],
        "Covered precision":    m["covered_precision"],
        "Covered recall":       m["covered_recall"],
        "Covered F1":           m["covered_f1"],
    }
 
report = pd.DataFrame({"TEST (full-train model)": _flow(m_test), "TRAIN OOF": _flow(m_oof)})
print("=" * 80)
print(f"📊 STAGE 2 — GLASS ROUTER REPORT  (profile: {CONFIG_PROFILE})")
print("=" * 80)
print(f"Rules (full-train model): Pass 1 = {m_test['n_pass1_rules']} | Pass 2 = {m_test['n_pass2_rules']}")
_fr = glass.fit_report_
print(f"Pass 2 population: {_fr.get('pass2_population')}", end="")
if _fr.get("pass2_population") == "oof_remainder":
    print(f" | remainder {_fr['remainder_rows']:,} rows ({_fr['remainder_fraction']:.1%}), "
          f"base rate {_fr['remainder_base_rate']:.3f}, "
          f"inner Pass 1 rule counts {_fr['inner_pass1_rule_counts']}")
else:
    print()
print(report.map(lambda v: f"{v:.1%}").to_string())
 
leak_oof = _flow(m_oof)["Pass 1 leakage"]
print(f"\nPass 1 operating constraint (OOF): leakage {leak_oof:.1%} vs target "
      f"{PASS1_LEAKAGE_TARGET:.0%} -> {'PASS' if leak_oof <= PASS1_LEAKAGE_TARGET else 'FAIL'}")
 
# Optimism Stage 4 would inherit from in-sample train features
glass_opt = oof_vs_insample(
    y_train, stage2_oof.glass_proba[:, 1],
    glass.predict_proba(GLASS_ROUTER_DATA["X_eng_train"])[:, 1],
)
rf_opt = oof_vs_insample(
    y_train, stage2_oof.rf_proba,
    rf_result.pipe.predict_proba(GLASS_ROUTER_DATA["X_eng_train"])[:, 1],
)
print("\nTrain AUC of P(subscribe): OOF vs in-sample")
print(pd.DataFrame({"GLASS Router": glass_opt, "RF": rf_opt}).round(4).to_string())
 
print("\nPer-fold cross-fit summary:")
print(stage2_oof.summary_frame().round(3).to_string(index=False))
 
# Test classification report (abstains counted as NOT_SUBSCRIBE)
p_all = test_out["pred"].copy()
p_all[test_out["abstained"]] = 0
print("\nTest classification report (abstain -> No Sub):")
print(classification_report(GLASS_ROUTER_DATA["y_eng_test"], p_all, target_names=["No Sub", "Sub"]))
 
print("Selected rules (full-train model):")
print(glass.get_rules_dataframe().round(3).to_string(index=False))

## Cell 16 — Save Stage 2 Artifacts

Saves the finalized RF and GLASS Router artifacts with the provenance needed for Stage 4.

- Records the shared train/test split fingerprint.
- Stores RF OOF probabilities, fold IDs, and protocol metadata.
- Saves GLASS OOF outputs and full-train test predictions.
- Retains in-sample GLASS probabilities for diagnostics only.
- Verifies artifact shapes, split alignment, and OOF provenance.

In [ ]:
%%legacy_stage2
# ============================================================
# CELL 16: SAVE STAGE 2 ARTIFACTS (RF + GLASS ROUTER)
# ============================================================
from glass_pipeline.glass_router.rf import save_rf_artifact
from glass_pipeline.glass_router.model_saver import ModelSaver
 
# Same fingerprint function the RF router records (sha256 of train/test row
# indices), so GLASS, RF and RF-router artifacts can be cross-checked.
from blackbox_pipeline.models.rf_router import split_fingerprint as router_split_fingerprint
SPLIT_FINGERPRINT = router_split_fingerprint(GLOBAL_SPLIT["X_train"], GLOBAL_SPLIT["X_test"])
print(f"split_fingerprint: {SPLIT_FINGERPRINT}")
 
import joblib
 
rf_saved = save_rf_artifact(rf_result, GLASS_ROUTER_DATA, output_dir="./models/rf")
 
# Add split provenance + OOF RF scores to the RF artifact (rf package untouched)
if not stage2_oof.index.equals(GLASS_ROUTER_DATA["X_eng_train"].index):
    raise ValueError("OOF block is not in X_eng_train order.")
rf_art = joblib.load(rf_saved["path"])
rf_art.update({
    "split_fingerprint": SPLIT_FINGERPRINT,
    "index_test":        GLASS_ROUTER_DATA["X_eng_test"].index,
    "train_outputs_oof": True,
    "oof_proba":         stage2_oof.rf_proba,      
    "oof_index":         stage2_oof.index,
    "oof_fold_id":       stage2_oof.fold_id,
    "oof_protocol":      stage2_oof.protocol,
})
joblib.dump(rf_art, rf_saved["path"])
 
saver = ModelSaver(glass_model=glass)
glass_path = saver.save_model(
    train_out=None,                  # train side comes from the OOF block
    train_proba=None,
    oof=stage2_oof,                  # sets train_outputs_oof=True + fold ids + protocol
    train_insample_proba=glass.predict_proba(GLASS_ROUTER_DATA["X_eng_train"]),  
    test_out=test_out,
    test_proba=test_proba,
    global_split=GLOBAL_SPLIT,
    split_fingerprint=SPLIT_FINGERPRINT,
    y_test=GLASS_ROUTER_DATA["y_eng_test"],
)
 
print(f"\n📦 RF artifact    → {rf_saved['path']}")
print(f"📦 GLASS artifact → {glass_path}")

## Cell 17 — Stage 3 Feature Block

Builds the Stage 3 feature block — the same block the black-box XGBoost arm builds — or reuses a cached copy that is provably the block that would be built now.

- Applies the per-fold GLASS feature pipeline.
- Enforces the shared feature, fold, and split contract.
- Reuses the cache only when its recorded identity matches the current data, split, fold and feature settings, feature-engineering code and library versions, and the cached block passes its own checks (inner folds, content fingerprint, settings, rows); otherwise rebuilds it and replaces the cache (`shared.stage3.block_cache`, PR 36 F23).
- Verifies train/test alignment and feature parity.

In [ ]:
# ============================================================
# CELL 17: STAGE 3 FEATURE BLOCK — per-fold GLASS DAG (validated cache)
# ============================================================
from pathlib import Path
from shared.stage3.feature_engineering import EBM_FEATURES, EBMFeaturePipeline 
from glass_pipeline.ebm import EBMStage3Config
import hashlib
import json
from shared.stage3.block_cache import block_identity, load_or_build_block
from shared import split_fingerprint

# The GLASS EBM config (PR 36 F21): it labels the artifact model_family="ebm",
# arm="glass" and records the EBM search space; train_ebm_stage refuses the
# generic Stage3Config.
EBM_STAGE3_CONFIG = EBMStage3Config(
    expected_features=list(EBM_FEATURES),
    n_trials=150,                    # fair comparison -> XGBoost arm's budget
    artifact_dir="models/ebm",
    verbose=True,
)

_FP_NOW = split_fingerprint(GLOBAL_SPLIT["X_train"].index, GLOBAL_SPLIT["X_test"].index,
                            GLOBAL_SPLIT["y_train"], GLOBAL_SPLIT["y_test"])

# Reused only when the cache is provably the block that would be built now (data, split,
# fold/feature settings, feature-engineering code, library versions) and passes the block's
# own checks; otherwise rebuilt and saved atomically (PR 36 F23). The cache name carries a
# digest of that identity (PR 39, execution contract §7.1): a block built by other code or
# data is written beside the existing cache, never over it.
_BLOCK_ID = hashlib.sha256(json.dumps(
    block_identity(GLOBAL_SPLIT, EBMFeaturePipeline, EBM_STAGE3_CONFIG, _FP_NOW),
    sort_keys=True, default=str).encode("utf-8")).hexdigest()[:12]
_BLOCK_CACHE = Path("models/stage3") / f"stage3_block_{_FP_NOW}_{_BLOCK_ID}.joblib"
STAGE3_BLOCK, _block_status = load_or_build_block(
    GLOBAL_SPLIT, EBMFeaturePipeline, EBM_STAGE3_CONFIG,
    split_fingerprint=_FP_NOW, cache_path=_BLOCK_CACHE,
)
print(f"block {_block_status}: {_BLOCK_CACHE.name}")

assert STAGE3_BLOCK.features == list(EBM_FEATURES)
assert STAGE3_BLOCK.index_train.equals(GLOBAL_SPLIT["X_train"].index)
assert STAGE3_BLOCK.index_test.equals(GLOBAL_SPLIT["X_test"].index)
print(f"block fingerprint: {STAGE3_BLOCK.block_fingerprint}")

## Cell 18 — Stage 3 EBM

Trains the GLASS Stage 3 EBM using the shared Stage 3 protocol.

- Uses the shared feature block and 150-trial tuning budget.
- Selects the operating threshold from OOF predictions.
- Applies calibration when required by the calibration gate.
- Evaluates once on the held-out test split.
- Exposes OOF train and refit test frames for Stage 4.

Stage 4 trains on `STAGE3_EBM_TRAIN` and evaluates on `STAGE3_EBM_TEST`.

In [ ]:
# ============================================================
# CELL 18: STAGE 3 — EBM
# ============================================================
from glass_pipeline.ebm import train_ebm_stage


print("=" * 80)
print("🌳 STAGE 3: EBM (shared Stage 3 protocol)")
print("=" * 80)

ebm_artifact, ebm_path = train_ebm_stage(block=STAGE3_BLOCK, config=EBM_STAGE3_CONFIG)

_m = ebm_artifact.metrics["test_at_operating_threshold"]
_c = ebm_artifact.calibration
print(f"\n✅ EBM Stage 3 complete")
print(f"📦 Saved artifact → {ebm_path}")
print(f"📊 Threshold (OOF-selected): {ebm_artifact.threshold['oof']['threshold']:.3f}")
print(f"📊 Test ROC-AUC: {_m['roc_auc']:.4f}")
print(f"📊 Test F2:      {_m['f2']:.4f}   F1: {_m['f1']:.4f}")
print(f"📊 ECE OOF  {_c['ece_before']:.4f} → {_c['ece_after']:.4f}  "
      f"({'calibrated' if _c['applied'] else 'not calibrated'})")
print(f"📊 ECE test {_c['test_ece_raw']:.4f} → {_c['test_ece_calibrated']:.4f}")

# Stage 4 inputs 
STAGE3_EBM_TRAIN = ebm_artifact.stage4_frame()          # OOF — train Stage 4 on this
STAGE3_EBM_TEST = ebm_artifact.stage4_test_frame()      # refit — evaluate Stage 4
STAGE3_EBM_TRAIN_META = ebm_artifact.stage4_metadata()  # fold_id / threshold / state — NOT features

## Cell 19 — Stage 4: GLASS Arbiter (weighted-confidence arbiter)

> **Naming:** the GLASS Arbiter (`glass_pipeline.glass_arbiter`) is a **weighted-confidence arbiter**. Through PR 38 it was `meta_ebm` / "Meta-EBM"; PR 39 renamed it with no behavioral change, and artifacts saved before that (`meta_ebm/2`, `models/meta_ebm/`) remain readable.

Combines LR, GLASS Router, and EBM predictions, abstaining when confidence is too low.

**Inputs:** OOF train predictions and held-out test predictions from all three stages. `shared/stage4` verifies split, indices, labels, folds, router semantics, and leakage safety before fitting.

**Procedure:**

1. Set LR/EBM thresholds for 70% recall; router stays at 0.5.
2. Compute trust weights from Brier score and accuracy.
3. Measure disagreement and correctness.
4. Tune the abstention cut on OOF train predictions for F2 with ≥50% coverage.
5. Evaluate once on test data.

**Arbitration:** each stage contributes threshold-relative confidence × trust weight. LR and EBM always vote; the router votes only when it routes. Low-confidence rows abstain.

The artifact keeps upstream thresholds, arbiter thresholds, and the abstention cut separate.

**Output:** `meta_artifact` (`glass_arbiter/1`) with provenance, weights, thresholds, predictions, and validation results, saved to `models/glass_arbiter/`.


In [ ]:
# ============================================================
# CELL 19: GLASS ARBITER — MODULAR ARBITER
# ============================================================
from glass_pipeline.glass_arbiter.meta_stage import train_glass_arbiter

meta_artifact, meta_path = train_glass_arbiter(
    GLOBAL_SPLIT,
    lr_path=lr_calibrated_path,
    glass_path=glass_path,
    ebm_path=ebm_path,
    target_recall=0.70,
)

## Cell 20 — Cascade Trace

Analyzes target overlap across the Stage 1 LR, Stage 2 GLASS Router, and Stage 3 EBM.

- Measures shared and model-unique positive catches.
- Identifies redundant vs complementary cascade behavior.
- Reports the hard-error ceiling where no stage catches the target.
- Saves a Venn diagram for the research log.

In [ ]:
# ============================================================
# CELL 20: VENN DIAGRAM TRACE — CASCADE (LR + EBM + Glass Router)
# ============================================================
from glass_pipeline.glass_arbiter.tracer import run_cascade_trace

trace = run_cascade_trace(
    GLOBAL_SPLIT=GLOBAL_SPLIT,
    meta_artifact=meta_artifact,
    figures_dir="research_logs/figures",
)